# 04 - Golden Record Creation: FDA Medical Device Master Data
## Project: Healthcare Device Master Data Governance
### Objective
Apply survivorship rules to create a single trusted golden record for each 
master data entity. This notebook covers:
1. **Applicant Master** — one golden record per unique company
2. **Device Master** — one golden record per unique device
3. **Output** — clean master dataset saved to data/cleaned/

Golden record creation is the core deliverable of any MDM initiative — 
producing a single, trusted, authoritative version of master data entities.

In [26]:
# Import libraries and load data
import pandas as pd
import numpy as np
from rapidfuzz import fuzz, process
import re

# Load dataset
df = pd.read_csv('/Users/harshithasunkara/Documents/Projects/Healthcare MDM/data/raw/pmn96cur.txt',
                 sep='|',
                 encoding='latin-1',
                 low_memory=False)

# Load duplicate candidates from notebook 3
duplicates = pd.read_csv('/Users/harshithasunkara/Documents/Projects/Healthcare MDM/reports/duplicate_candidates.csv')

print(f"Master dataset: {df.shape[0]:,} records")
print(f"Duplicate candidates to resolve: {len(duplicates):,}")

Master dataset: 99,216 records
Duplicate candidates to resolve: 117


In [27]:
# ================================
# STEP 1: SURVIVORSHIP RULES
# ================================
print("=== APPLYING SURVIVORSHIP RULES ===")

# Rule 1: Prefer most frequently occurring name (more submissions = more established)
# Rule 2: Prefer properly cased names (not ALL CAPS)

def pick_survivor(name1, name2, df):
    count1 = df[df['APPLICANT'] == name1].shape[0]
    count2 = df[df['APPLICANT'] == name2].shape[0]
    
    # Rule 1: Pick most frequently occurring name
    if count1 != count2:
        return name1 if count1 > count2 else name2
    
    # Rule 2: Pick properly cased (not all caps)
    if name1.isupper() and not name2.isupper():
        return name2
    if name2.isupper() and not name1.isupper():
        return name1
    
    return name1

# Apply survivorship to high confidence duplicates only
high_conf = duplicates[duplicates['CONFIDENCE'] == 'HIGH'].copy()

# Build mapping table
mapping = {}
for _, row in high_conf.iterrows():
    survivor = pick_survivor(row['NAME_1'], row['NAME_2'], df)
    loser = row['NAME_2'] if survivor == row['NAME_1'] else row['NAME_1']
    mapping[loser] = survivor

print(f"Survivorship rules applied to {len(high_conf)} high confidence pairs")
print(f"\nSample mappings (loser → golden record):")
for k, v in list(mapping.items())[:10]:
    print(f"  '{k}' → '{v}'")

=== APPLYING SURVIVORSHIP RULES ===
Survivorship rules applied to 26 high confidence pairs

Sample mappings (loser → golden record):
  'sa instruments' → 'usa instruments'
  'biocoll laboratories' → 'biocell laboratories'
  'medtronic sofamor danck usa' → 'medtronic sofamor danek usa'
  'srs medical' → 'rs medical'
  'qrs medical' → 'rs medical'
  'acon laboratories' → 'alcon laboratories'
  'depuy orthopedics' → 'depuy orthopaedics'
  'ultradent product' → 'ultradent products'
  'biomet manufactuting' → 'biomet manufacturing'
  'biomet manafacturing' → 'biomet manufacturing'


In [28]:
# Verify SRS Medical vs RS Medical
print("SRS Medical records:")
print(df[df['APPLICANT'].str.contains('Srs Medical', case=False, na=False)]['APPLICANT'].value_counts())

print("\nRS Medical records:")
print(df[df['APPLICANT'].str.contains('^Rs Medical$', case=False, na=False)]['APPLICANT'].value_counts())

SRS Medical records:
APPLICANT
Srs Medical Systems, Inc.    3
Srs Medical                  1
Name: count, dtype: int64

RS Medical records:
APPLICANT
Rs Medical    7
Name: count, dtype: int64


### Survivorship Rule Limitation
Frequency-based survivorship can sometimes pick the wrong record if the 
incorrect name appears more often than the correct one. In enterprise MDM,
this would be flagged for data steward review rather than auto-merged.
Example: 'ciriticare systems' (typo) winning over 'criticare systems' (correct)
demonstrates why human oversight is essential in MDM workflows.

In [29]:
# See ALL 117 duplicate pairs completely
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
print(duplicates.to_string())

                                          NAME_1                                    NAME_2  SIMILARITY_SCORE CONFIDENCE
0                                usa instruments                            sa instruments         96.551724       HIGH
1                              inova diagnostics                        genova diagnostics         91.428571     MEDIUM
2                              life technologies                      lifeloc technologies         91.891892     MEDIUM
3                              life technologies                          led technologies         90.909091     MEDIUM
4                          acist medical systems                       cas medical systems         90.000000     MEDIUM
5                          acist medical systems                       is2 medical systems         90.000000     MEDIUM
6                              nobel biocare uas                         nobel biocare usa         94.117647     MEDIUM
7                           medist inter

In [30]:
print(df[df['APPLICANT'].str.contains('Medpro|Med Pro', case=False, na=False)]['APPLICANT'].value_counts())

print(df[df['APPLICANT'].str.contains('DMC Medical|DM Medical', case=False, na=False)]['APPLICANT'].value_counts())

print(df[df['APPLICANT'].str.contains('Advanced Technology Research', case=False, na=False)]['APPLICANT'].value_counts())



APPLICANT
Medpro Safety Products, Inc.            3
Changzhou Holymed Products Co., Ltd.    3
Medpro Technologies, Inc.               1
Safety-Med Products, Inc.               1
Sunsmed Protective Products , Ltd.      1
In-Med Prognostics L3c                  1
Biomed Products                         1
Name: count, dtype: int64
APPLICANT
Dmc Medical, Ltd.     3
Dm Medical, Inc.      1
Padm Medical, Inc.    1
Bdm Medical, Inc.     1
Name: count, dtype: int64
APPLICANT
Advanced Technology Research (A.T.R.) S.R.P.    2
Advanced Technology Research (A.T.R.) S.R.L.    2
Name: count, dtype: int64


In [31]:
# ================================
# FINAL VERIFIED MAPPING TABLE
# (auto-generated mapping + manual corrections + false positive removals)
# ================================
mapping = {
    'sa instruments': 'usa instruments',
    'biocoll laboratories': 'biocell laboratories',
    'medtronic sofamor danck usa': 'medtronic sofamor danek usa',
    'acon laboratories': 'alcon laboratories',
    'depuy orthopedics': 'depuy orthopaedics',
    'ultradent product': 'ultradent products',
    'biomet manufactuting': 'biomet manufacturing',
    'biomet manafacturing': 'biomet manufacturing',
    'ciriticare systems': 'criticare systems',
    'phillips medical systems cleveland': 'philips medical systems cleveland',
    'med pro technologies': 'medpro technologies',
    'fresensius medical care north america': 'fresenius medical care north america',
    'medtronic soafamor danek': 'medtronic sofamor danek',
    'medtrade product': 'medtrade products',
    'smith newphew': 'smith nephew',
    'ge medical syst information technologies': 'ge medical systems information technologies',
    'baxterhealthcare': 'baxter healthcare',
    'merit medical system': 'merit medical systems',
    'arrow int l': 'arrow intl',
    'abott laboratories': 'abbott laboratories',
    'monlycke health care': 'molnlycke health care',
}

print(f"Final verified mapping: {len(mapping)} entries")
for k, v in mapping.items():
    print(f"  '{k}' → '{v}'")

Final verified mapping: 21 entries
  'sa instruments' → 'usa instruments'
  'biocoll laboratories' → 'biocell laboratories'
  'medtronic sofamor danck usa' → 'medtronic sofamor danek usa'
  'acon laboratories' → 'alcon laboratories'
  'depuy orthopedics' → 'depuy orthopaedics'
  'ultradent product' → 'ultradent products'
  'biomet manufactuting' → 'biomet manufacturing'
  'biomet manafacturing' → 'biomet manufacturing'
  'ciriticare systems' → 'criticare systems'
  'phillips medical systems cleveland' → 'philips medical systems cleveland'
  'med pro technologies' → 'medpro technologies'
  'fresensius medical care north america' → 'fresenius medical care north america'
  'medtronic soafamor danek' → 'medtronic sofamor danek'
  'medtrade product' → 'medtrade products'
  'smith newphew' → 'smith nephew'
  'ge medical syst information technologies' → 'ge medical systems information technologies'
  'baxterhealthcare' → 'baxter healthcare'
  'merit medical system' → 'merit medical systems'
 

In [32]:
# ================================
# STEP 2: APPLY MAPPING TO DATASET
# ================================
print("=== CREATING GOLDEN RECORDS ===")

df_golden = df.copy()

def normalize_name(name):
    if pd.isna(name):
        return ''
    name = str(name).lower().strip()
    suffixes = [r',?\s+inc\.?$', r',?\s+llc\.?$', r',?\s+ltd\.?$',
                r',?\s+corp\.?$', r',?\s+corporation$', r',?\s+incorporated$',
                r',?\s+co\.?$', r',?\s+gmbh$', r',?\s+lp$', r',?\s+l\.p\.$']
    for suffix in suffixes:
        name = re.sub(suffix, '', name)
    name = re.sub(r'[^\w\s]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

df_golden['APPLICANT_NORMALIZED'] = df_golden['APPLICANT'].apply(normalize_name)
df_golden['APPLICANT_GOLDEN'] = df_golden['APPLICANT_NORMALIZED'].replace(mapping)

before = df_golden['APPLICANT_NORMALIZED'].nunique()
after = df_golden['APPLICANT_GOLDEN'].nunique()
print(f"Unique applicants before: {before:,}")
print(f"Unique applicants after: {after:,}")
print(f"Records standardized: {before - after:,}")

# Verify key fixes
checks = ['criticare systems', 'fresenius medical care north america',
          'philips medical systems cleveland', 'medpro technologies']
print(f"\nVerification:")
for name in checks:
    count = (df_golden['APPLICANT_GOLDEN'] == name).sum()
    print(f"  '{name}': {count:,} records")

=== CREATING GOLDEN RECORDS ===
Unique applicants before: 21,769
Unique applicants after: 21,748
Records standardized: 21

Verification:
  'criticare systems': 10 records
  'fresenius medical care north america': 44 records
  'philips medical systems cleveland': 90 records
  'medpro technologies': 2 records


In [33]:
# Verify ALL 21 mappings applied correctly
print("=== VERIFYING ALL 21 MAPPINGS ===")
for bad_name, good_name in mapping.items():
    bad_count = (df_golden['APPLICANT_GOLDEN'] == bad_name).sum()
    good_count = (df_golden['APPLICANT_GOLDEN'] == good_name).sum()
    status = "PASS" if bad_count == 0 else "FAIL - still exists"
    print(f"{status} | '{bad_name}' -> '{good_name}' ({good_count} records)")

=== VERIFYING ALL 21 MAPPINGS ===
PASS | 'sa instruments' -> 'usa instruments' (65 records)
PASS | 'biocoll laboratories' -> 'biocell laboratories' (3 records)
PASS | 'medtronic sofamor danck usa' -> 'medtronic sofamor danek usa' (184 records)
PASS | 'acon laboratories' -> 'alcon laboratories' (157 records)
PASS | 'depuy orthopedics' -> 'depuy orthopaedics' (209 records)
PASS | 'ultradent product' -> 'ultradent products' (72 records)
PASS | 'biomet manufactuting' -> 'biomet manufacturing' (131 records)
PASS | 'biomet manafacturing' -> 'biomet manufacturing' (131 records)
PASS | 'ciriticare systems' -> 'criticare systems' (10 records)
PASS | 'phillips medical systems cleveland' -> 'philips medical systems cleveland' (90 records)
PASS | 'med pro technologies' -> 'medpro technologies' (2 records)
PASS | 'fresensius medical care north america' -> 'fresenius medical care north america' (44 records)
PASS | 'medtronic soafamor danek' -> 'medtronic sofamor danek' (254 records)
PASS | 'medtrade

In [34]:
# ================================
# STEP 3: BUILD FINAL MASTER DATASET
# ================================
print("=== BUILDING FINAL MASTER DATASET ===")

cols_to_drop = ['SSPINDICATOR', 'EXPEDITEDREVIEW']
df_golden = df_golden.drop(columns=cols_to_drop)

df_golden['MDM_RECORD_TYPE'] = 'GOLDEN'
df_golden['MDM_SOURCE'] = 'FDA_510K'
df_golden['MDM_PROCESSED_DATE'] = pd.Timestamp.today().strftime('%Y-%m-%d')
df_golden['MDM_NAME_STANDARDIZED'] = df_golden['APPLICANT_NORMALIZED'] != df_golden['APPLICANT_GOLDEN']

final_cols = ['KNUMBER', 'APPLICANT', 'APPLICANT_GOLDEN',
              'DEVICENAME', 'PRODUCTCODE', 'DECISION',
              'DATERECEIVED', 'DECISIONDATE', 'CITY',
              'STATE', 'COUNTRY_CODE', 'MDM_RECORD_TYPE',
              'MDM_SOURCE', 'MDM_PROCESSED_DATE', 'MDM_NAME_STANDARDIZED']

df_final = df_golden[final_cols]

print(f"Final master dataset shape: {df_final.shape}")
print(f"Records with standardized names: {df_final['MDM_NAME_STANDARDIZED'].sum():,}")
print(f"\nSample golden records:")
print(df_final[df_final['MDM_NAME_STANDARDIZED'] == True].head(5)[
    ['KNUMBER', 'APPLICANT', 'APPLICANT_GOLDEN']].to_string())

=== BUILDING FINAL MASTER DATASET ===
Final master dataset shape: (99216, 15)
Records with standardized names: 123

Sample golden records:
      KNUMBER                   APPLICANT     APPLICANT_GOLDEN
595   K000714  Med-Pro Technologies, Inc.  medpro technologies
2895  K003557     ACON Laboratories, Inc.   alcon laboratories
3770  K010582       Acon Laboratories Co.   alcon laboratories
3990  K010841     ACON Laboratories, Inc.   alcon laboratories
4424  K011353     ACON Laboratories, Inc.   alcon laboratories


In [35]:
# ================================
# PROPER CASING FOR GOLDEN RECORDS
# ================================
# Title case the golden record name for professional output
df_final = df_final.copy()
df_final['APPLICANT_GOLDEN'] = df_final['APPLICANT_GOLDEN'].str.title()

print("Sample after proper casing:")
print(df_final[df_final['MDM_NAME_STANDARDIZED'] == True].head(5)[
    ['APPLICANT', 'APPLICANT_GOLDEN']].to_string())

Sample after proper casing:
                       APPLICANT     APPLICANT_GOLDEN
595   Med-Pro Technologies, Inc.  Medpro Technologies
2895     ACON Laboratories, Inc.   Alcon Laboratories
3770       Acon Laboratories Co.   Alcon Laboratories
3990     ACON Laboratories, Inc.   Alcon Laboratories
4424     ACON Laboratories, Inc.   Alcon Laboratories


In [36]:
# ================================
# STEP 4: SAVE GOLDEN MASTER DATASET
# ================================
output_path = '/Users/harshithasunkara/Documents/Projects/Healthcare MDM/data/cleaned/fda_device_master_golden.csv'
df_final.to_csv(output_path, index=False)

print("=== GOLDEN RECORD CREATION COMPLETE ===")
print(f"Total master records: {len(df_final):,}")
print(f"Records standardized: {df_final['MDM_NAME_STANDARDIZED'].sum():,}")
print(f"Columns dropped (empty): 2 (SSPINDICATOR, EXPEDITEDREVIEW)")
print(f"MDM metadata columns added: 3 (MDM_RECORD_TYPE, MDM_SOURCE, MDM_PROCESSED_DATE)")
print(f"Stewardship review queue: 91 medium confidence pairs saved to reports/")
print(f"Golden master dataset saved to: data/cleaned/fda_device_master_golden.csv")

=== GOLDEN RECORD CREATION COMPLETE ===
Total master records: 99,216
Records standardized: 123
Columns dropped (empty): 2 (SSPINDICATOR, EXPEDITEDREVIEW)
MDM metadata columns added: 3 (MDM_RECORD_TYPE, MDM_SOURCE, MDM_PROCESSED_DATE)
Stewardship review queue: 91 medium confidence pairs saved to reports/
Golden master dataset saved to: data/cleaned/fda_device_master_golden.csv
